In [38]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tokenizer

BertTokenizerFast(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [39]:
!wget https://archive.ics.uci.edu/static/public/19/car+evaluation.zip
!unzip car+evaluation.zip

--2025-04-20 18:13:59--  https://archive.ics.uci.edu/static/public/19/car+evaluation.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘car+evaluation.zip’

car+evaluation.zip      [ <=>                ]   6.19K  --.-KB/s    in 0s      

2025-04-20 18:13:59 (127 MB/s) - ‘car+evaluation.zip’ saved [6342]

Archive:  car+evaluation.zip
  inflating: car.c45-names           
  inflating: car.data                
  inflating: car.names               


In [40]:
# !unzip census+income.zip

In [41]:
!pip install openml

In [42]:
import pandas as pd

df = pd.read_csv('car.data', sep=',', names=['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class'])
print(len(df))
df

1728


,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc
...,...,...,...,...,...,...,...
1723,low,low,5more,more,med,med,good
1724,low,low,5more,more,med,high,vgood
1725,low,low,5more,more,big,low,unacc
1726,low,low,5more,more,big,med,good


In [43]:
df.columns

Index(['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class'], dtype='object')

In [44]:
import numpy as np
import torch

DROP_P = 0.2

def concatenate_text(x_full):
    x = {}
    for i, j in x_full.items():
        x[i] = j if np.random.random() > DROP_P else '[UNK]'
    
    if x['buying'] == 'vhigh':
        b = 'very high'
    elif x['buying'] == 'high':
        b = 'high'
    elif x['buying'] == 'med':
        b = 'medium'
    elif x['buying'] == 'low':
        b = 'low'
    else:
        b = '[UNK]'

    if x['maint'] == 'vhigh':
        m = 'very high'
    elif x['maint'] == 'high':
        m = 'high'
    elif x['maint'] == 'med':
        m = 'medium'
    elif x['maint'] == 'low':
        m = 'low'
    else:
        m = '[UNK]'

    if x['doors'] == '5more':
        d = '5 or more'
    elif x['doors'] == '[UNK]':
        d = '[UNK]'
    else:
        d = x['doors']

    if x['persons'] == 'more':
        p = 'more than 4'
    elif x['persons'] == '[UNK]':
        p = '[UNK]'
    else:
        p = x['persons']

    if x['lug_boot'] == 'med':
        l = 'medium'
    elif x['lug_boot'] == '[UNK]':
        l = '[UNK]'
    else:
        l = x['lug_boot']

    if x['safety'] == 'med':
        s = 'medium'
    elif x['safety'] == '[UNK]':
        s = '[UNK]'
    else:
        s = x['safety']

    
    text = "".join([f"I have information about a car. ",
            f"The buying price of it is {b}. ",
            f"The maintenance price of it is {m}. ",
            f"It has {d} doors. ",
            f"It has places for {p} people. ",
            f"The size of luggage boot is {l}. ",
            f"The safety of the car is estimated as {s}."])

    return text

concatenate_text(df.iloc[0])

'I have information about a car. The buying price of it is very high. The maintenance price of it is very high. It has 2 doors. It has places for 2 people. The size of luggage boot is [UNK]. The safety of the car is estimated as [UNK].'

In [45]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.drop('class', axis =1),
                                                    df['class'],
                                                    test_size=.2,
                                                    random_state = 42)
y_train = y_train.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})
y_test = y_test.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})

X_train['text'] = X_train.apply(lambda x: concatenate_text(x), axis=1)
X_test['text'] = X_test.apply(lambda x: concatenate_text(x), axis=1)

X_train['label'] = y_train
X_test['label'] = y_test

<ipython-input-45-37af98c5fa93>:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_train = y_train.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})
<ipython-input-45-37af98c5fa93>:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_test = y_test.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})


In [46]:
X_train['text'].iloc[0]

'I have information about a car. The buying price of it is very high. The maintenance price of it is very high. It has 5 or more doors. It has places for [UNK] people. The size of luggage boot is big. The safety of the car is estimated as high.'

In [47]:
len(X_train)

1382

In [48]:
!pip install evaluate

In [49]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import numpy as np
import evaluate

# Define label mappings
# id2label = {0: "NOT-DONATE", 1: "DONATE"}
# label2id = {"NOT-DONATE": 0, "DONATE": 1}

# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(X_train)
test_dataset = Dataset.from_pandas(X_test)

def tokenize_function(examples):
    # Adjust based on the structure of your dataset
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

# Format the datasets correctly with labels
tokenized_train_dataset = tokenized_train_dataset.map(lambda x: {'labels': x['label']})
tokenized_test_dataset = tokenized_test_dataset.map(lambda x: {'labels': x['label']})

Map:   0%|          | 0/1382 [00:00<?, ? examples/s]

Map:   0%|          | 0/346 [00:00<?, ? examples/s]

Map:   0%|          | 0/1382 [00:00<?, ? examples/s]

Map:   0%|          | 0/346 [00:00<?, ? examples/s]

In [50]:
tokenized_train_dataset[0].keys()

dict_keys(['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'text', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'])

In [51]:
tokenized_train_dataset[0]['text']

'I have information about a car. The buying price of it is very high. The maintenance price of it is very high. It has 5 or more doors. It has places for [UNK] people. The size of luggage boot is big. The safety of the car is estimated as high.'

In [52]:
tokenized_train_dataset[0]['label']

0

In [53]:
from transformers import BertForSequenceClassification
from transformers import DataCollatorWithPadding
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from sklearn.metrics import roc_curve, auc
import sklearn.metrics as metrics
import matplotlib.pyplot as plt
import sklearn.metrics as metrics

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4).to('cuda')

model.dropout.p = 0

for param in model.bert.parameters():
    param.requires_grad = False

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    warmup_steps=0,
    learning_rate = 0.01,
    weight_decay=0.001,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    report_to='none'
)

# Evaluation metric
auc = evaluate.load("roc_auc", 'multiclass')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # print(logits)
    predictions = torch.softmax(torch.tensor(logits), dim=1).numpy()
    return auc.compute(prediction_scores=predictions, references=labels, multi_class='ovr')

# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# Train the model
trainer.train()

results = trainer.evaluate()
print(results)

pred = trainer.predict(tokenized_test_dataset)
print("test f1", f1_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test precision", precision_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test recall", recall_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test accuracy", accuracy_score(y_test, np.argmax(pred[0], 1)))
# fpr, tpr, threshold = roc_curve(y_test, pred[0][:, 1])
# roc_auc = metrics.auc(fpr, tpr)
# print("test roc_auc", roc_auc)
# print("")

pred_train = trainer.predict(tokenized_train_dataset)
print("train f1", f1_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train precision", precision_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train recall", recall_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train accuracy", accuracy_score(y_train, np.argmax(pred_train[0], 1)))

# fpr_train, tpr_train, threshold_train = roc_curve(y_train, pred_train[0][:, 1])
# roc_auc_train = metrics.auc(fpr_train, tpr_train)
# print("train roc_auc", roc_auc_train)


# plt.title('Receiver Operating Characteristic')
# plt.plot(fpr, tpr, 'b', label = 'AUC = %0.2f' % roc_auc)
# plt.legend(loc = 'lower right')
# plt.plot([0, 1], [0, 1],'r--')
# plt.xlim([0, 1])
# plt.ylim([0, 1])
# plt.ylabel('True Positive Rate')
# plt.xlabel('False Positive Rate')
# plt.show()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Roc Auc
1,1.013800,1.110487,0.541260
2,0.960800,1.185351,0.621961
3,0.943100,0.973538,0.650889
4,0.999800,0.965802,0.678041
5,0.918300,0.984867,0.687758
6,0.830700,0.858402,0.666641
7,0.830900,1.008096,0.690061
8,0.847900,0.874783,0.674553
9,0.796200,0.862846,0.676262
10,0.925000,0.849180,0.679810


{'eval_loss': 0.849179744720459, 'eval_roc_auc': 0.679810454134573, 'eval_runtime': 1.2856, 'eval_samples_per_second': 269.133, 'eval_steps_per_second': 4.667, 'epoch': 10.0}
test f1 0.5494314168316535
test precision 0.4613000768485415
test recall 0.6791907514450867
test accuracy 0.6791907514450867


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


train f1 0.5836756847693877
train precision 0.49772922901644256
train recall 0.7054992764109985
train accuracy 0.7054992764109985


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [54]:
for param in model.bert.parameters():
    param.requires_grad = True

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=20,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    learning_rate = 0.00001,
    weight_decay=0.001,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    report_to='none'
)

# # Evaluation metric
# auc = evaluate.load("roc_auc")

# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     predictions = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
#     return auc.compute(prediction_scores=predictions, references=labels)

# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# Train the model
trainer.train()

results = trainer.evaluate()
print(results)

pred = trainer.predict(tokenized_test_dataset)
print("test f1", f1_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test precision", precision_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test recall", recall_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test accuracy", accuracy_score(y_test, np.argmax(pred[0], 1)))
# fpr, tpr, threshold = roc_curve(y_test, pred[0][:, 1])
# roc_auc = metrics.auc(fpr, tpr)
# print("test roc_auc", roc_auc)
# print("")

pred_train = trainer.predict(tokenized_train_dataset)
print("train f1", f1_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train precision", precision_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train recall", recall_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train accuracy", accuracy_score(y_train, np.argmax(pred_train[0], 1)))

# fpr_train, tpr_train, threshold_train = roc_curve(y_train, pred_train[0][:, 1])
# roc_auc_train = metrics.auc(fpr_train, tpr_train)
# print("train roc_auc", roc_auc_train)


# plt.title('Receiver Operating Characteristic')
# plt.plot(fpr, tpr, 'b', label = 'AUC = %0.2f' % roc_auc)
# plt.legend(loc = 'lower right')
# plt.plot([0, 1], [0, 1],'r--')
# plt.xlim([0, 1])
# plt.ylim([0, 1])
# plt.ylabel('True Positive Rate')
# plt.xlabel('False Positive Rate')
# plt.show()

Epoch,Training Loss,Validation Loss,Roc Auc
1,0.821000,0.847462,0.692149
2,0.800500,0.846753,0.724744
3,0.806600,0.829600,0.762965
4,0.778000,0.799095,0.801062
5,0.755800,0.727152,0.840557
6,0.647700,0.638618,0.812024
7,0.602600,0.568804,0.884003
8,0.564300,0.544473,0.885743
9,0.503800,0.507808,0.901582
10,0.614100,0.496456,0.909480


{'eval_loss': 0.5471190214157104, 'eval_roc_auc': 0.9424294054157625, 'eval_runtime': 1.2842, 'eval_samples_per_second': 269.425, 'eval_steps_per_second': 4.672, 'epoch': 20.0}
test f1 0.7281525303576586
test precision 0.756390122897218
test recall 0.7687861271676301
test accuracy 0.7687861271676301


train f1 0.7830762184425261
train precision 0.817722846278995
train recall 0.8125904486251809
train accuracy 0.8125904486251809


In [57]:
DROP_P = 0.5

X_train, X_test, y_train, y_test = train_test_split(df.drop('class', axis =1),
                                                    df['class'],
                                                    test_size=.2,
                                                    random_state = 42)
y_train = y_train.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})
y_test = y_test.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})

X_train['text'] = X_train.apply(lambda x: concatenate_text(x), axis=1)
X_test['text'] = X_test.apply(lambda x: concatenate_text(x), axis=1)

X_train['label'] = y_train
X_test['label'] = y_test

train_dataset = Dataset.from_pandas(X_train)
test_dataset = Dataset.from_pandas(X_test)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

# Format the datasets correctly with labels
tokenized_train_dataset = tokenized_train_dataset.map(lambda x: {'labels': x['label']})
tokenized_test_dataset = tokenized_test_dataset.map(lambda x: {'labels': x['label']})

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4).to('cuda')

model.dropout.p = 0

for param in model.bert.parameters():
    param.requires_grad = False

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    warmup_steps=0,
    learning_rate = 0.01,
    weight_decay=0.001,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    report_to='none'
)

# Evaluation metric
auc = evaluate.load("roc_auc", 'multiclass')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # print(logits)
    predictions = torch.softmax(torch.tensor(logits), dim=1).numpy()
    return auc.compute(prediction_scores=predictions, references=labels, multi_class='ovr')

# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# Train the model
trainer.train()

results = trainer.evaluate()
print(results)

pred = trainer.predict(tokenized_test_dataset)
print("test f1", f1_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test precision", precision_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test recall", recall_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test accuracy", accuracy_score(y_test, np.argmax(pred[0], 1)))

pred_train = trainer.predict(tokenized_train_dataset)
print("train f1", f1_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train precision", precision_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train recall", recall_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train accuracy", accuracy_score(y_train, np.argmax(pred_train[0], 1)))

<ipython-input-57-9abaf45bdb06>:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_train = y_train.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})
<ipython-input-57-9abaf45bdb06>:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_test = y_test.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})


Map:   0%|          | 0/1382 [00:00<?, ? examples/s]

Map:   0%|          | 0/346 [00:00<?, ? examples/s]

Map:   0%|          | 0/1382 [00:00<?, ? examples/s]

Map:   0%|          | 0/346 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss,Roc Auc
1,0.987200,0.946815,0.503588
2,0.917900,0.917942,0.535837
3,0.914000,1.002935,0.532138
4,0.941500,1.008898,0.574378
5,0.977800,1.160951,0.615287
6,0.806000,0.881411,0.635171
7,0.817500,0.961232,0.636130
8,0.873500,0.871221,0.661799
9,0.807600,0.869398,0.662678
10,0.942200,0.858387,0.660109


{'eval_loss': 0.8583873510360718, 'eval_roc_auc': 0.6601088710982443, 'eval_runtime': 1.2805, 'eval_samples_per_second': 270.201, 'eval_steps_per_second': 4.686, 'epoch': 10.0}
test f1 0.5494314168316535
test precision 0.4613000768485415
test recall 0.6791907514450867
test accuracy 0.6791907514450867


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


train f1 0.5836756847693877
train precision 0.49772922901644256
train recall 0.7054992764109985
train accuracy 0.7054992764109985


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [58]:
for param in model.bert.parameters():
    param.requires_grad = True

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=20,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    learning_rate = 0.00001,
    weight_decay=0.001,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    report_to='none'
)


# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# Train the model
trainer.train()

results = trainer.evaluate()
print(results)

pred = trainer.predict(tokenized_test_dataset)
print("test f1", f1_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test precision", precision_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test recall", recall_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test accuracy", accuracy_score(y_test, np.argmax(pred[0], 1)))

pred_train = trainer.predict(tokenized_train_dataset)
print("train f1", f1_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train precision", precision_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train recall", recall_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train accuracy", accuracy_score(y_train, np.argmax(pred_train[0], 1)))

Epoch,Training Loss,Validation Loss,Roc Auc
1,0.834000,0.858830,0.660885
2,0.813000,0.864860,0.664372
3,0.823700,0.856555,0.674570
4,0.807800,0.849521,0.686429
5,0.810400,0.836479,0.723091
6,0.764900,0.809920,0.768571
7,0.735600,0.757093,0.786538
8,0.724500,0.710148,0.780615
9,0.678600,0.703561,0.777972
10,0.776200,0.688042,0.784853


{'eval_loss': 0.6510598659515381, 'eval_roc_auc': 0.8467841592335532, 'eval_runtime': 1.2881, 'eval_samples_per_second': 268.609, 'eval_steps_per_second': 4.658, 'epoch': 20.0}
test f1 0.6582752465147909
test precision 0.6416971154891352
test recall 0.7254335260115607
test accuracy 0.7254335260115607


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


train f1 0.7031299329196561
train precision 0.6760793198925157
train recall 0.7547033285094067
train accuracy 0.7547033285094067


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [59]:
DROP_P = 0.9

X_train, X_test, y_train, y_test = train_test_split(df.drop('class', axis =1),
                                                    df['class'],
                                                    test_size=.2,
                                                    random_state = 42)
y_train = y_train.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})
y_test = y_test.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})

X_train['text'] = X_train.apply(lambda x: concatenate_text(x), axis=1)
X_test['text'] = X_test.apply(lambda x: concatenate_text(x), axis=1)

X_train['label'] = y_train
X_test['label'] = y_test

train_dataset = Dataset.from_pandas(X_train)
test_dataset = Dataset.from_pandas(X_test)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

# Format the datasets correctly with labels
tokenized_train_dataset = tokenized_train_dataset.map(lambda x: {'labels': x['label']})
tokenized_test_dataset = tokenized_test_dataset.map(lambda x: {'labels': x['label']})

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4).to('cuda')

model.dropout.p = 0

for param in model.bert.parameters():
    param.requires_grad = False

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    warmup_steps=0,
    learning_rate = 0.01,
    weight_decay=0.001,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    report_to='none'
)

# Evaluation metric
auc = evaluate.load("roc_auc", 'multiclass')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # print(logits)
    predictions = torch.softmax(torch.tensor(logits), dim=1).numpy()
    return auc.compute(prediction_scores=predictions, references=labels, multi_class='ovr')

# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# Train the model
trainer.train()

results = trainer.evaluate()
print(results)

pred = trainer.predict(tokenized_test_dataset)
print("test f1", f1_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test precision", precision_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test recall", recall_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test accuracy", accuracy_score(y_test, np.argmax(pred[0], 1)))

pred_train = trainer.predict(tokenized_train_dataset)
print("train f1", f1_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train precision", precision_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train recall", recall_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train accuracy", accuracy_score(y_train, np.argmax(pred_train[0], 1)))

<ipython-input-59-bac728d27978>:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_train = y_train.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})
<ipython-input-59-bac728d27978>:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_test = y_test.replace({'unacc':0, 'acc':1, 'vgood':2, 'good':3})


Map:   0%|          | 0/1382 [00:00<?, ? examples/s]

Map:   0%|          | 0/346 [00:00<?, ? examples/s]

Map:   0%|          | 0/1382 [00:00<?, ? examples/s]

Map:   0%|          | 0/346 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss,Roc Auc
1,1.207000,1.129265,0.446187
2,1.057100,1.058837,0.471520
3,1.405700,1.216345,0.484009
4,0.960900,1.028481,0.534409
5,0.959600,1.052419,0.490837
6,0.807000,0.953108,0.479375
7,0.813800,0.990261,0.483118
8,0.881200,0.887336,0.562326
9,0.814200,0.878360,0.542604
10,0.946400,0.865907,0.544348


{'eval_loss': 0.8659066557884216, 'eval_roc_auc': 0.544348090053162, 'eval_runtime': 1.2878, 'eval_samples_per_second': 268.686, 'eval_steps_per_second': 4.659, 'epoch': 10.0}
test f1 0.5494314168316535
test precision 0.4613000768485415
test recall 0.6791907514450867
test accuracy 0.6791907514450867


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


train f1 0.5836756847693877
train precision 0.49772922901644256
train recall 0.7054992764109985
train accuracy 0.7054992764109985


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [60]:
for param in model.bert.parameters():
    param.requires_grad = True

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=20,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    learning_rate = 0.00001,
    weight_decay=0.001,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    report_to='none'
)


# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# Train the model
trainer.train()

results = trainer.evaluate()
print(results)

pred = trainer.predict(tokenized_test_dataset)
print("test f1", f1_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test precision", precision_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test recall", recall_score(y_test, np.argmax(pred[0], 1), average='weighted'))
print("test accuracy", accuracy_score(y_test, np.argmax(pred[0], 1)))

pred_train = trainer.predict(tokenized_train_dataset)
print("train f1", f1_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train precision", precision_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train recall", recall_score(y_train, np.argmax(pred_train[0], 1), average='weighted'))
print("train accuracy", accuracy_score(y_train, np.argmax(pred_train[0], 1)))

Epoch,Training Loss,Validation Loss,Roc Auc
1,0.844400,0.864903,0.532858
2,0.830700,0.867781,0.538327
3,0.831900,0.868074,0.526434
4,0.814300,0.869452,0.498314
5,0.830500,0.868100,0.539450
6,0.795500,0.871838,0.558741
7,0.798900,0.874153,0.540635
8,0.832000,0.868022,0.541362
9,0.803900,0.872254,0.528363
10,0.938300,0.869411,0.538750


{'eval_loss': 0.8789049983024597, 'eval_roc_auc': 0.5386420504471666, 'eval_runtime': 1.2913, 'eval_samples_per_second': 267.951, 'eval_steps_per_second': 4.647, 'epoch': 20.0}
test f1 0.5494314168316535
test precision 0.4613000768485415
test recall 0.6791907514450867
test accuracy 0.6791907514450867


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


train f1 0.5836756847693877
train precision 0.49772922901644256
train recall 0.7054992764109985
train accuracy 0.7054992764109985


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
